
# Privacy-Conscious Patient Data Processor

**Week 1 Internship Milestone | Python Fundamentals**

## Objective
Build a Python-based pipeline that reads synthetic patient records,
masks personally identifiable information (PII), validates records,
and exports clean records to a new CSV file.

## Features
- Synthetic patient dataset
- CSV reading and writing
- Patient name redaction
- Phone number masking
- Data validation
- Separate valid and invalid records

**Privacy note:** This project uses fictional data.
Basic PII masking does not guarantee complete anonymization.


In [4]:
# Saving the Data in csv File
%%writefile patients.csv
PatientID,Name,Age,PhoneNumber,Diagnosis
001,John Doe,45,555-0198,Diabetes
002,Jane Smith,32,555-0234,Asthma
003,Alice Johnson,,555-0371,Hypertension
004,Bob Brown,150,555-0455,Flu
005,Charlie Davis,28,invalid-phone,Sprained Ankle
006,Diana Prince,30,555-0101,Migraine
007,Evan Wright,55,555-0202,Arthritis
008,Fiona Gallagher,-5,555-0303,Checkup
009,George Martin,68,555-0404,
010,Hannah Abbott,22,555-0505,Anemia
011,Ian Malcolm,44,,Broken Arm
012,Julia Child,88,555-0707,Hypertension
013,Kevin Bacon,62,555-0808,High Cholesterol
014,,35,555-0909,Insomnia
015,Maria Garcia,40,555-1010,Diabetes
016,Noah Smith,12,555-1111,Asthma
017,Olivia Pope,37,555-1212,Stress
018,Peter Parker,18,555-1313,Spider Bite
019,Quinn Fabray,17,555-1414,Checkup
020,Rachel Green,29,555-1515,Pregnancy
021,Steve Rogers,105,555-1616,Fatigue
022,Tony Stark,48,555-1717,Heart Condition
023,Ursula Sea,55,5551818,Laryngitis
024,Victor Stone,24,555-1919,Physical Therapy
025,Wanda Maximoff,30,555-2020,Headache
026,Xavier Charles,60,555-2121,Back Pain
027,Yara Greyjoy,33,555-2222,Scurvy
028,Zack Taylor,16,555-2323,Concussion
029,Alan Turing,41,555-2424,Allergy
030,Bruce Wayne,35,missing,Bruises

Overwriting patients.csv


In [5]:
import csv

patients = []

with open("patients.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        patients.append(row)

print("Total records:", len(patients))
print("First 3 records:")

for patient in patients[:3]:
    print(patient)



Total records: 30
First 3 records:
{'PatientID': '001', 'Name': 'John Doe', 'Age': '45', 'PhoneNumber': '555-0198', 'Diagnosis': 'Diabetes'}
{'PatientID': '002', 'Name': 'Jane Smith', 'Age': '32', 'PhoneNumber': '555-0234', 'Diagnosis': 'Asthma'}
{'PatientID': '003', 'Name': 'Alice Johnson', 'Age': '', 'PhoneNumber': '555-0371', 'Diagnosis': 'Hypertension'}


In [6]:
# PII redation from synthatic data
masked_patients = []

for patient in patients:
    safe_patient = patient.copy()
    safe_patient['Name'] = '[REDACTED]'
    phone = safe_patient['PhoneNumber']

    if len(phone) >= 4:
        safe_patient["PhoneNumber"] = "XXX-XXX-" + phone[-4:]
    else:
        safe_patient["PhoneNumber"] = "[REDACTED]"

    masked_patients.append(safe_patient)

print("First 3 Masked Patients Data")
for p in masked_patients[:3]:
    print(p)


First 3 Masked Patients Data
{'PatientID': '001', 'Name': '[REDACTED]', 'Age': '45', 'PhoneNumber': 'XXX-XXX-0198', 'Diagnosis': 'Diabetes'}
{'PatientID': '002', 'Name': '[REDACTED]', 'Age': '32', 'PhoneNumber': 'XXX-XXX-0234', 'Diagnosis': 'Asthma'}
{'PatientID': '003', 'Name': '[REDACTED]', 'Age': '', 'PhoneNumber': 'XXX-XXX-0371', 'Diagnosis': 'Hypertension'}



## 1. Data Validation

Before exporting records, check the following:

- Patient ID must contain exactly 3 digits.
- Name cannot be empty.
- Age must be a whole number between 1 and 119.
- Phone number must follow the format `555-0198`.
- Diagnosis cannot be empty.

Invalid records are collected separately with explanations.

Validation takes place before masking so malformed phone numbers
can be detected rather than accidentally accepted.


In [7]:

import re

valid_patients = []
invalid_patients = []

for patient in patients:
    errors = []

    # Validate patient ID
    if not re.fullmatch(r"\d{3}", patient["PatientID"].strip()):
        errors.append("Invalid Patient ID")

    # Validate name
    if not patient["Name"].strip():
        errors.append("Missing name")

    # Validate age
    age = patient["Age"].strip()

    if not age:
        errors.append("Missing age")
    else:
        try:
            age_num = int(age)

            if not 1 <= age_num <= 119:
                errors.append("Unrealistic age")

        except ValueError:
            errors.append("Age must be a whole number")

    # Validate phone number
    phone = patient["PhoneNumber"].strip()

    if not re.fullmatch(r"\d{3}-\d{4}", phone):
        errors.append("Invalid phone number")

    # Validate diagnosis
    if not patient["Diagnosis"].strip():
        errors.append("Missing diagnosis")

    if errors:
        invalid_patients.append({
            "PatientID": patient["PatientID"],
            "Errors": errors
        })
    else:
        valid_patients.append(patient.copy())

print("Valid records:", len(valid_patients))
print("Invalid records:", len(invalid_patients))

for record in invalid_patients:
    print(record)


Valid records: 21
Invalid records: 9
{'PatientID': '003', 'Errors': ['Missing age']}
{'PatientID': '004', 'Errors': ['Unrealistic age']}
{'PatientID': '005', 'Errors': ['Invalid phone number']}
{'PatientID': '008', 'Errors': ['Unrealistic age']}
{'PatientID': '009', 'Errors': ['Missing diagnosis']}
{'PatientID': '011', 'Errors': ['Invalid phone number']}
{'PatientID': '014', 'Errors': ['Missing name']}
{'PatientID': '023', 'Errors': ['Invalid phone number']}
{'PatientID': '030', 'Errors': ['Invalid phone number']}



## 2. Personally Identifiable Information (PII) Masking

PII masking reduces exposure of sensitive information.

Our processing rules:

- Replace patient names with `[REDACTED]`.
- Hide the first three phone digits.
- Keep the last four phone digits.
- Preserve patient ID, age, and diagnosis.

Example:

**Original:** John Doe, 555-0198

**Masked:** [REDACTED], XXX-0198

This is partial masking, not complete anonymization.


In [8]:
masked_patients = []

for patient in valid_patients:
    safe_patient = patient.copy()

    safe_patient["Name"] = "[REDACTED]"

    phone = patient["PhoneNumber"]
    safe_patient["PhoneNumber"] = "XXX-" + phone[-4:]

    masked_patients.append(safe_patient)

print("First 3 masked records:")

for patient in masked_patients[:3]:
    print(patient)


First 3 masked records:
{'PatientID': '001', 'Name': '[REDACTED]', 'Age': '45', 'PhoneNumber': 'XXX-0198', 'Diagnosis': 'Diabetes'}
{'PatientID': '002', 'Name': '[REDACTED]', 'Age': '32', 'PhoneNumber': 'XXX-0234', 'Diagnosis': 'Asthma'}
{'PatientID': '006', 'Name': '[REDACTED]', 'Age': '30', 'PhoneNumber': 'XXX-0101', 'Diagnosis': 'Migraine'}


In [9]:

headers = [
    "PatientID",
    "Name",
    "Age",
    "PhoneNumber",
    "Diagnosis"
]

with open(
    "clean_patients.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(masked_patients)

print("Clean patient records exported successfully.")
print("Records exported:", len(masked_patients))


Clean patient records exported successfully.
Records exported: 21


In [10]:

with open("clean_patients.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    exported_records = list(reader)

print("Exported records:", len(exported_records))

assert len(exported_records) == len(valid_patients)

for record in exported_records:
    assert record["Name"] == "[REDACTED]"
    assert re.fullmatch(r"XXX-\d{4}", record["PhoneNumber"])

print("All exported records passed the masking checks.")


Exported records: 21
All exported records passed the masking checks.



## 3. Results

The pipeline processes 30 synthetic patient records.

It:
- Checks required fields and input formats.
- Identifies invalid records with error messages.
- Masks names and phone numbers in valid records.
- Exports validated, masked records to `clean_patients.csv`.
- Preserves the original synthetic source file.

## 4. Limitations

- Validation rules are simplified for demonstration purposes.
- Only one phone number format is accepted.
- The allowed age range is an application-specific rule.
- Patient IDs and medical attributes remain in the output.
- Masking does not guarantee anonymization or regulatory compliance.
- The project has not been evaluated for real patient data.

## 5. Conclusion

This milestone demonstrates Python fundamentals, including CSV
processing, loops, dictionaries, conditionals, exception handling,
regular expressions, and basic privacy-conscious data processing.



## What I Learned

- Applied Python fundamentals such as loops, conditionals, lists, and dictionaries.
- Learned to read, process, and write CSV files using the `csv` module.
- Implemented data validation to identify missing and invalid records.
- Used regular expressions to validate phone numbers and patient IDs.
- Applied PII masking to protect patient names and phone numbers.
- Used exception handling to manage invalid data.
- Preserved original data while exporting cleaned records.
- Used assertions to verify the processed output.
- Understood the difference between data masking and anonymization.

**Key Takeaway:** Learned to build a basic privacy-conscious data processing pipeline using core Python.
